# HET quick-look presentation playground

This notebook has a short operational path followed by optional evidence
inspection. Configure the raw and trace roots once, inspect the exposure
inventory, select a row, and make a quick look. The package retains the
detector, trace, topology, extraction, amplifier, and channel evidence for
diagnostic work below.


## 1. Configure the session

`QuicklookSite` expands the paths once and keeps the instrument-specific
discovery configuration for the notebook session. The package must be
installed, preferably in editable mode while developing it.


In [ ]:
from hetquicklook import QuicklookSite

ql = QuicklookSite(
    raw_roots={
        "lrs2": "~/data/LRS2",
        "virus": "~/data/VIRUS",
    },
)


## 2. Choose a date and inspect the exposure inventory

The table has one row per encoded exposure ID, including multiple exposure IDs
inside one observation archive. Rows use zero-based Python indexing.


In [ ]:
night = ql.night("20260512", instrument="lrs2")
night


## 3. Select one exposure and make the quick look

Classification chooses the flat or standard-star workflow automatically. An
unsupported exposure raises a descriptive error rather than selecting a
workflow by guesswork.


In [ ]:
# Example recognized standard-star row for the date above; choose another row from the table as needed.
exposure = night[25]
exposure


In [ ]:
product = exposure.quicklook()


In [ ]:
product.plot()


## Advanced / diagnostic inspection

The cells below are optional. They use the objects retained by the normal
workflow; they do not repeat detector reduction, topology construction, or
LRS2 channel composition.


In [ ]:
print("Underlying observation:", exposure.observation)
print("Archive:", exposure.observation.archive_path)
print("Raw exposure:", exposure.raw_exposure)
print("Metadata:", exposure.metadata)
print("Classification:", exposure.classification)


In [ ]:
if product.instrument.value == "lrs2":
    print("Channels:", tuple(product.channels))
    print("Available amplifiers:", tuple(product.amplifier_evidence))
    print("Missing amplifiers:", product.missing_amplifiers)
    print("Processing failures:", product.processing_failures)
else:
    print("VIRUS IFUs:", tuple(product.ifus))
    print("Available amplifiers:", tuple(product.amplifier_evidence))
    print("Missing amplifiers:", product.missing_amplifiers)
    print("Processing failures:", product.processing_failures)


In [ ]:
# One amplifier's complete evidence tree.
if product.amplifier_evidence:
    token = next(iter(product.amplifier_evidence))
    amplifier_evidence = product.amplifier_evidence[token]
    print("Selected amplifier:", token)
    print("Raw provenance:", amplifier_evidence.loaded.provenance)
    print("Detector result:", amplifier_evidence.detector)
    print("Physical identity:", amplifier_evidence.topology.physical_identity)
    print("Trace provenance:", amplifier_evidence.topology.trace_provenance)
    print("Position provenance:", amplifier_evidence.topology.position_provenance)
    print("Spatial product:", amplifier_evidence.product)
else:
    print("No amplifier evidence was produced.")


In [ ]:
if product.amplifier_evidence:
    from hetquicklook.visualization import (
        plot_fiber_values,
        plot_spatial_image,
        plot_spatial_support,
    )

    amplifier_product = amplifier_evidence.product
    plot_spatial_image(amplifier_product)
    plot_fiber_values(amplifier_product)
    plot_spatial_support(amplifier_product)
else:
    print("No amplifier product is available.")


### VIRUS IFU selection

A VIRUS exposure can contain several IFUs. The high-level product keeps them
as a collection and never chooses one arbitrarily. When more than one is
present, select the physical IFU explicitly for its amplifier evidence view.


In [ ]:
if product.instrument.value == "virus":
    print("Available IFUs:", tuple(product.ifus))
    # Example:
    # product.plot(ifu="074")


### Presentation experiments

Display choices can be changed without changing the stored scientific
results. For example, a different percentile stretch can be used for
inspection or a figure can be saved after reviewing it.


In [ ]:
if product.instrument.value == "lrs2" or len(product.ifus) == 1:
    experiment_figure = product.plot(
        percentiles=(1.0, 99.0),
        show_fibers=True,
    )
    # experiment_figure.savefig("quicklook_experiment.png", dpi=160, bbox_inches="tight")
else:
    print("Select a VIRUS IFU before plotting a multi-IFU exposure.")


The lower-level workflow APIs remain available for tests and
specialized investigations. Use them when a prepared detector array and
amplifier-level `FiberTopology` are the objects of interest.
